In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 01_ingest_bronze - Creación de Capa Bronze
# MAGIC Ingestar CSV de Olist a formato Delta/Parquet

# COMMAND ----------

from datetime import datetime

# Configuración - Rutas completas con catalog/schema/volume
INPUT_PATH = "/Volumes/olist/olist_csv/olist2/"
OUTPUT_PATH = "/Volumes/olist/olist_bronze/bronze/"  # Ruta completa con schema y volume

# Mapeo: nombre_archivo_csv -> nombre_tabla_bronze
TABLES = {
    "olist_customers_dataset.csv": "customers",
    "olist_orders_dataset.csv": "orders",
    "olist_order_items_dataset.csv": "order_items",
    "olist_order_payments_dataset.csv": "order_payments",
    "olist_order_reviews_dataset.csv": "order_reviews",
    "olist_products_dataset.csv": "products",
    "olist_sellers_dataset.csv": "sellers",
    "olist_geolocation_dataset.csv": "geolocation",
    "product_category_name_translation.csv": "product_category_translation"
}

start_time = datetime.now()
print(f"🚀 Inicio: {start_time.strftime('%H:%M:%S')}")
print(f"📂 Input:  {INPUT_PATH}")
print(f"📂 Output: {OUTPUT_PATH}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Crear Volume Bronze (si no existe)

# COMMAND ----------

# Verificar y crear el schema y volume si no existen
print("🔧 Verificando estructura de Unity Catalog...\n")

try:
    # Crear schema si no existe
    spark.sql("CREATE SCHEMA IF NOT EXISTS olist.olist_bronze")
    print("✅ Schema 'olist.olist_bronze' verificado/creado")
    
    # Crear volume si no existe
    spark.sql("""
        CREATE VOLUME IF NOT EXISTS olist.olist_bronze.bronze
        COMMENT 'Capa Bronze - Datos raw de Olist'
    """)
    print("✅ Volume 'bronze' verificado/creado")
    print(f"✅ Ruta completa: {OUTPUT_PATH}\n")
    
except Exception as e:
    print(f"⚠️  Advertencia al crear estructura: {e}")
    print("Intentando continuar con la ruta existente...\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Ingesta de Datos

# COMMAND ----------

# Procesar cada tabla
results = []

for csv_file, table_name in TABLES.items():
    try:
        print(f"📥 {table_name} ({csv_file})... ", end="")
        
        # Leer CSV
        df = spark.read.csv(f"{INPUT_PATH}{csv_file}", header=True, inferSchema=True)
        rows = df.count()
        cols = len(df.columns)
        
        # Guardar en Delta (fallback a Parquet si falla)
        output = f"{OUTPUT_PATH}{table_name}/"
        try:
            df.write.format("delta").mode("overwrite").save(output)
            fmt = "Delta"
        except:
            df.write.mode("overwrite").parquet(output)
            fmt = "Parquet"
        
        print(f"✅ {rows:,} filas, {cols} cols [{fmt}]")
        
        # Mostrar esquema resumido
        schema_preview = ', '.join([f'{c.name}({c.dataType.simpleString()})' for c in df.schema.fields[:3]])
        print(f"   Esquema: {schema_preview}...")
        
        results.append({"table": table_name, "csv": csv_file, "rows": rows, "cols": cols, "status": "OK"})
        
    except Exception as e:
        print(f"❌ Error: {e}")
        results.append({"table": table_name, "csv": csv_file, "rows": 0, "cols": 0, "status": "ERROR"})

# COMMAND ----------

# MAGIC %md
# MAGIC ## Resumen de Resultados

# COMMAND ----------

# Resumen
print(f"\n{'='*70}")
print("📊 RESUMEN")
print(f"{'='*70}")

success = [r for r in results if r["status"] == "OK"]
errors = [r for r in results if r["status"] == "ERROR"]

print(f"✅ Exitosas: {len(success)}/{len(TABLES)}")
print(f"❌ Errores: {len(errors)}/{len(TABLES)}\n")

if success:
    print(f"{'Tabla Bronze':<30} {'Filas':>12} {'Columnas':>10}")
    print("-" * 55)
    for r in success:
        print(f"{r['table']:<30} {r['rows']:>12,} {r['cols']:>10}")
    print("-" * 55)
    print(f"{'TOTAL':<30} {sum(r['rows'] for r in success):>12,}")

if errors:
    print(f"\n⚠️  Tablas con errores:")
    for r in errors:
        print(f"  • {r['table']} (archivo: {r['csv']})")

# Tiempo
duration = (datetime.now() - start_time).total_seconds()
print(f"\n⏱️  Duración: {duration:.2f} seg")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Verificación de Archivos

# COMMAND ----------

# Verificar archivos creados
print(f"\n📁 Estructura Bronze creada:")
try:
    items = sorted(dbutils.fs.ls(OUTPUT_PATH), key=lambda x: x.name)
    for item in items:
        if item.isDir():
            print(f"  ✓ /{item.name}")
except Exception as e:
    print(f"  ⚠️  Error al listar: {e}")

# COMMAND ----------

# MAGIC %md
# MAGIC ---
# MAGIC **Estructura creada:**
# MAGIC ```
# MAGIC Catalog: olist
# MAGIC └── Schema: olist_bronze
# MAGIC     └── Volume: bronze
# MAGIC         ├── customers/
# MAGIC         ├── orders/
# MAGIC         ├── order_items/
# MAGIC         ├── order_payments/
# MAGIC         ├── order_reviews/
# MAGIC         ├── products/
# MAGIC         ├── sellers/
# MAGIC         ├── geolocation/
# MAGIC         ├── product_category_translation/
# MAGIC ```